In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

class MultiScaleDilatedEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        # 1. Load STANDARD ResNet18 without the built-in dilation argument
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        resnet = models.resnet18(weights=weights)
        
        # 2. Modify for Grayscale CT Scans
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        if pretrained:
            self.conv1.weight.data = resnet.conv1.weight.data.mean(dim=1, keepdim=True)
            
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        # 3. Extract the layers
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # 4. The Custom Fix: Manually apply dilation to Layer 3 and 4
        # This prevents the image from shrinking while preserving pretrained weights!
        self._apply_dilation(self.layer3, dilation=2)
        self._apply_dilation(self.layer4, dilation=4)

    def _apply_dilation(self, layer, dilation):
        """Iterates through a ResNet BasicBlock and manually injects dilation."""
        for m in layer.modules():
            if isinstance(m, nn.Conv2d):
                # If it's a standard 3x3 convolution, add dilation and remove the stride
                if m.kernel_size == (3, 3):
                    m.stride = (1, 1)
                    m.padding = (dilation, dilation)
                    m.dilation = (dilation, dilation)
                # If it's the 1x1 downsampling convolution in the residual connection, just remove the stride
                elif m.kernel_size == (1, 1):
                    m.stride = (1, 1)

    def forward(self, x):
        features = []
        
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x1 = self.layer1(x)
        features.append(x1)
        
        x2 = self.layer2(x1)
        features.append(x2)
        
        x3 = self.layer3(x2)
        features.append(x3)
        
        x4 = self.layer4(x3)
        features.append(x4)
        
        return features

# --- Quick Test ---
if __name__ == "__main__":
    # Create a dummy batch of 2 grayscale images, 256x256
    dummy_input = torch.randn(2, 1, 256, 256) 
    
    encoder = MultiScaleDilatedEncoder(pretrained=True)
    outputs = encoder(dummy_input)
    
    print("Patched Encoder Output Feature Map Shapes:")
    for i, out in enumerate(outputs):
        print(f"Layer {i+1}: {out.shape} -> Channels: {out.shape[1]}, Resolution: {out.shape[2]}x{out.shape[3]}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 323MB/s]


Patched Encoder Output Feature Map Shapes:
Layer 1: torch.Size([2, 64, 64, 64]) -> Channels: 64, Resolution: 64x64
Layer 2: torch.Size([2, 128, 32, 32]) -> Channels: 128, Resolution: 32x32
Layer 3: torch.Size([2, 256, 32, 32]) -> Channels: 256, Resolution: 32x32
Layer 4: torch.Size([2, 512, 32, 32]) -> Channels: 512, Resolution: 32x32


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        # MLP shared by both AvgPool and MaxPool
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_pool = self.mlp(F.adaptive_avg_pool2d(x, 1).view(b, c))
        max_pool = self.mlp(F.adaptive_max_pool2d(x, 1).view(b, c))
        # Add the two pooled features and apply sigmoid
        return torch.sigmoid(avg_pool + max_pool).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Standard spatial attention uses a 7x7 conv to look at local pixel neighborhoods
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        # Concatenate average and max pooling along the channel dimension
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return torch.sigmoid(self.conv(x_cat))

class FrequencyAttention(nn.Module):
    """
    The novel FFT-based feature attention.
    Converts features to the frequency domain, dynamically weights the frequencies
    to enhance high-frequency nodules/edges, and converts back.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        # We use a 1x1 conv to learn which frequency components matter most
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, max(1, channels // reduction), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(1, channels // reduction), channels, 1, bias=False)
        )

    def forward(self, x):
        # 1. Convert spatial features to frequency domain using Real FFT
        # norm='ortho' preserves signal energy during the transform
        x_fft = torch.fft.rfft2(x, norm='ortho')
        
        # 2. Extract the amplitude (magnitude) of the frequencies
        amplitude = torch.abs(x_fft)
        
        # 3. Generate attention map from the amplitude
        # We learn which frequencies (high vs low) to boost
        freq_weights = torch.sigmoid(self.mlp(amplitude))
        
        # 4. Modulate the original frequency representation
        x_fft_modulated = x_fft * freq_weights
        
        # 5. Inverse FFT back to the spatial domain
        x_out = torch.fft.irfft2(x_fft_modulated, s=(x.size(2), x.size(3)), norm='ortho')
        
        return x_out

class HybridAttentionBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.freq_attention = FrequencyAttention(channels, reduction)
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        # Step 1: Enhance structural edges and small nodules in the frequency domain
        freq_enhanced = self.freq_attention(x)
        
        # Step 2: Figure out *which* feature maps (channels) are most important
        ca_weight = self.channel_attention(freq_enhanced)
        ca_out = freq_enhanced * ca_weight
        
        # Step 3: Figure out *where* in the image the target is
        sa_weight = self.spatial_attention(ca_out)
        out = ca_out * sa_weight
        
        # Residual connection to preserve original feature flow and gradients
        return out + x

# --- Quick Test ---
if __name__ == "__main__":
    # Simulating the output from Layer 3 of our Dilated Encoder
    dummy_layer3_output = torch.randn(2, 256, 32, 32) 
    
    attention_block = HybridAttentionBlock(channels=256)
    enhanced_features = attention_block(dummy_layer3_output)
    
    print(f"Input Shape:  {dummy_layer3_output.shape}")
    print(f"Output Shape: {enhanced_features.shape}")

Input Shape:  torch.Size([2, 256, 32, 32])
Output Shape: torch.Size([2, 256, 32, 32])


In [3]:
import torch
import torch.nn as nn

# --- MODULE 4: Cross-Domain Normalization ---
class AdaIN_Augmentation(nn.Module):
    """
    Randomly swaps feature statistics (mean and std) within a batch during training.
    This neutralizes domain shifts (Dataset Bias) between the COVID and LIDC scanners.
    """
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p # Probability of applying the augmentation

    def calc_mean_std(self, feat, eps=1e-5):
        # Calculate spatial mean and standard deviation for each channel
        N, C = feat.size()[:2]
        feat_var = feat.view(N, C, -1).var(dim=2) + eps
        feat_std = feat_var.sqrt().view(N, C, 1, 1)
        feat_mean = feat.view(N, C, -1).mean(dim=2).view(N, C, 1, 1)
        return feat_mean, feat_std

    def forward(self, x):
        # Only apply during training, and only with probability `p`
        if not self.training or torch.rand(1).item() > self.p:
            return x
            
        # Shuffle the batch to pick random target styles
        idx = torch.randperm(x.size(0))
        target_feat = x[idx]
        
        # Extract statistics
        mean_x, std_x = self.calc_mean_std(x)
        mean_target, std_target = self.calc_mean_std(target_feat)
        
        # Normalize the input and apply the target style
        x_normalized = (x - mean_x) / std_x
        x_adain = x_normalized * std_target + mean_target
        
        return x_adain

# --- MODULE 3: Semantic Multi-Class Decoder ---
class DecoderBlock(nn.Module):
    """Standard Convolutional Block for the Decoder"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class LightweightMultiClassDecoder(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        
        # Cross-Domain handler
        self.adain = AdaIN_Augmentation(p=0.5)
        
        # 1. Deep Fusion Block (Combines Layer 4, 3, and 2 since they are all 32x32)
        # Channels: 512 (L4) + 256 (L3) + 128 (L2) = 896
        self.deep_fusion = DecoderBlock(in_channels=896, out_channels=256)
        
        # 2. Upsample to 64x64
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        
        # 3. Fuse with Layer 1 (64 channels) -> 128 + 64 = 192
        self.decode1 = DecoderBlock(in_channels=192, out_channels=128)
        
        # 4. Upsample to 128x128
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decode2 = DecoderBlock(in_channels=64, out_channels=64)
        
        # 5. Upsample to 256x256 (Original Image Size)
        self.up3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.decode3 = DecoderBlock(in_channels=32, out_channels=32)
        
        # 6. Final Pixel-wise Classifier (5 Classes)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, features):
        l1, l2, l3, l4 = features
        
        # Apply AdaIN to the deepest, most semantic layer to force domain invariance
        l4 = self.adain(l4)
        
        # Deep Multi-Scale Fusion (Concat along channel dimension)
        deep_cat = torch.cat([l4, l3, l2], dim=1)
        x = self.deep_fusion(deep_cat)
        
        # Upsample and fuse with Layer 1
        x = self.up1(x)
        x = torch.cat([x, l1], dim=1)
        x = self.decode1(x)
        
        # Upsample to 128x128
        x = self.up2(x)
        x = self.decode2(x)
        
        # Upsample to 256x256
        x = self.up3(x)
        x = self.decode3(x)
        
        # Output Logits for 5 classes
        out = self.final_conv(x)
        return out

# --- Quick Test ---
if __name__ == "__main__":
    # Simulate the outputs from our Dilated Encoder
    dummy_features = [
        torch.randn(2, 64, 64, 64),   # Layer 1
        torch.randn(2, 128, 32, 32),  # Layer 2
        torch.randn(2, 256, 32, 32),  # Layer 3
        torch.randn(2, 512, 32, 32)   # Layer 4
    ]
    
    decoder = LightweightMultiClassDecoder(num_classes=5)
    
    # Simulate Training Mode (so AdaIN activates)
    decoder.train() 
    
    segmentation_mask = decoder(dummy_features)
    print(f"Final Output Shape: {segmentation_mask.shape} -> Batch: {segmentation_mask.shape[0]}, Classes: {segmentation_mask.shape[1]}, Resolution: {segmentation_mask.shape[2]}x{segmentation_mask.shape[3]}")

Final Output Shape: torch.Size([2, 5, 256, 256]) -> Batch: 2, Classes: 5, Resolution: 256x256


In [4]:
import torch
import torch.nn as nn

class RobustLungSegNet(nn.Module):
    def __init__(self, num_classes=5, mc_dropout_rate=0.3):
        super().__init__()
        
        # MODULE 1: The Multi-Scale Dilated Encoder
        self.encoder = MultiScaleDilatedEncoder(pretrained=True)
        
        # MODULE 2: Hybrid Attention Blocks (Attached to the deep layers)
        self.attn2 = HybridAttentionBlock(channels=128)
        self.attn3 = HybridAttentionBlock(channels=256)
        self.attn4 = HybridAttentionBlock(channels=512)
        
        # MODULE 5: Monte Carlo Dropout
        self.mc_dropout = nn.Dropout2d(p=mc_dropout_rate)
        
        # MODULE 3 & 4: The Lightweight Decoder with AdaIN
        self.decoder = LightweightMultiClassDecoder(num_classes=num_classes)

    def forward(self, x):
        """Standard forward pass used during training."""
        # 1. Encode
        features = self.encoder(x)
        l1, l2, l3, l4 = features
        
        # 2. Apply Attention
        l2 = self.attn2(l2)
        l3 = self.attn3(l3)
        l4 = self.attn4(l4)
        
        # 3. Apply Dropout
        l2 = self.mc_dropout(l2)
        l3 = self.mc_dropout(l3)
        l4 = self.mc_dropout(l4)
        
        # 4. Decode
        out = self.decoder([l1, l2, l3, l4])
        return out

    def predict_with_uncertainty(self, x, num_passes=10):
        """
        Special inference method used during testing to generate 
        both the segmentation mask and the uncertainty map.
        """
        # Force the model into train mode so MC Dropout remains active
        self.train() 
        
        predictions = []
        with torch.no_grad():
            for _ in range(num_passes):
                pred_logits = self.forward(x)
                # Convert logits to probabilities (0.0 to 1.0)
                pred_probs = torch.softmax(pred_logits, dim=1) 
                predictions.append(pred_probs)
        
        # Stack all passes: Shape will be [passes, batch, classes, H, W]
        predictions = torch.stack(predictions)
        
        # 1. The final prediction is the average probability across all passes
        mean_prediction = predictions.mean(dim=0)
        
        # 2. The uncertainty map is the variance across all passes
        # We average the variance across the class dimension to get a single 2D heat map per image
        uncertainty_map = predictions.var(dim=0).mean(dim=1) 
        
        return mean_prediction, uncertainty_map

# --- Final Grand Test ---
if __name__ == "__main__":
    # Simulate a batch of 2 CT scans
    dummy_input = torch.randn(2, 1, 256, 256)
    
    # Initialize our complete, unified model!
    model = RobustLungSegNet(num_classes=5)
    
    # Test Standard Forward Pass (Training)
    train_output = model(dummy_input)
    print(f"Training Output Shape: {train_output.shape}")
    
    # Test Uncertainty Estimation (Testing/Inference)
    mean_pred, uncertainty = model.predict_with_uncertainty(dummy_input, num_passes=5)
    print(f"Mean Prediction Shape: {mean_pred.shape}")
    print(f"Uncertainty Map Shape: {uncertainty.shape}")

Training Output Shape: torch.Size([2, 5, 256, 256])
Mean Prediction Shape: torch.Size([2, 5, 256, 256])
Uncertainty Map Shape: torch.Size([2, 256, 256])
